In [2]:
# train_xgboost_v3 — XGBoost 二分类 + 多分类 + 特征重要性
# StratifiedGroupKFold (groups=Gene) + sample_weight

import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score
)
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings("ignore")

# ===== 加载数据 =====
df = pd.read_csv("/mnt/volume6/czj/labLGN/LabLZ/xgboost_trial/features_v3.csv")

# 特征列 (排除标识列 + 占位列)
ID_COLS = ["KEY", "Gene", "reloc_v3"]
DROP_COLS = ID_COLS + ["ddg", "plddt_diff"]
feat_cols = [c for c in df.columns if c not in DROP_COLS]

X = df[feat_cols].values.astype(np.float32)
y_5class = df["reloc_v3"].values.astype(int)
groups = df["Gene"].values

class_names = ["不重定位(C1)", "聚集(C2)", "分泌途径(C3)", "核定位(C4)", "细胞质(C5)"]

print(f"样本: {len(y_5class)}  特征: {X.shape[1]}  基因: {len(set(groups))}")
for i in range(5):
    print(f"  Class {i} ({class_names[i]:10s}): {(y_5class == i).sum()}")


样本: 2179  特征: 1288  基因: 871
  Class 0 (不重定位(C1)  ): 1944
  Class 1 (聚集(C2)    ): 13
  Class 2 (分泌途径(C3)  ): 34
  Class 3 (核定位(C4)   ): 121
  Class 4 (细胞质(C5)   ): 29


In [3]:
# ===== 手动 CV 工具 (支持每折 sample_weight) =====

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

def cv_predict_proba(X, y, groups, xgb_model):
    """手动 StratifiedGroupKFold，每折 fit 时传 sample_weight='balanced'"""
    n_classes = y.max() + 1
    oof = np.zeros((len(y), n_classes), dtype=np.float32)
    for fold, (tr_idx, te_idx) in enumerate(cv.split(X, y, groups)):
        X_tr, X_te = X[tr_idx], X[te_idx]
        y_tr = y[tr_idx]

        imp = SimpleImputer(strategy="median")
        scl = StandardScaler()
        X_tr = scl.fit_transform(imp.fit_transform(X_tr)).astype(np.float32)
        X_te = scl.transform(imp.transform(X_te)).astype(np.float32)

        sw = compute_sample_weight("balanced", y_tr)
        xgb_model.fit(X_tr, y_tr, sample_weight=sw, verbose=False)
        oof[te_idx] = xgb_model.predict_proba(X_te)
    return oof


In [4]:
# ===== 二分类: reloc vs no-reloc =====

y_bin = (y_5class > 0).astype(int)
print(f"二分类: 不重定位={(y_bin == 0).sum()}, 重定位={(y_bin == 1).sum()}\n")

xgb_bin = XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.5,
    objective="binary:logistic", eval_metric="aucpr",
    random_state=42, n_jobs=-1, tree_method="hist",
)

oof_bin = cv_predict_proba(X, y_bin, groups, xgb_bin)[:, 1]

print(f"Overall AUROC: {roc_auc_score(y_bin, oof_bin):.3f}")
print(f"Overall AUPRC: {average_precision_score(y_bin, oof_bin):.3f}")


二分类: 不重定位=1944, 重定位=235

Overall AUROC: 0.594
Overall AUPRC: 0.160


In [5]:
# ===== 多分类: 5-class =====

xgb_multi = XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.5,
    objective="multi:softprob", eval_metric="mlogloss",
    random_state=42, n_jobs=-1, tree_method="hist",
)

oof_5 = cv_predict_proba(X, y_5class, groups, xgb_multi)

print("Per-class AUROC (One-vs-Rest):")
for i in range(5):
    y_bin_i = (y_5class == i).astype(int)
    if y_bin_i.sum() > 0:
        auc = roc_auc_score(y_bin_i, oof_5[:, i])
        print(f"  Class {i} {class_names[i]:10s}: AUROC={auc:.3f}  (n={y_bin_i.sum()})")

oof_pred = oof_5.argmax(axis=1)
print(f"\nOverall accuracy: {(oof_pred == y_5class).mean():.3f}")
print(f"Macro F1: {f1_score(y_5class, oof_pred, average='macro'):.3f}")


KeyboardInterrupt: 

In [6]:
# ===== 特征重要性 (二分类模型 fit on all data) =====

imp = SimpleImputer(strategy="median")
scl = StandardScaler()
X_all = scl.fit_transform(imp.fit_transform(X)).astype(np.float32)

sw_ratio = (y_bin == 0).sum() / (y_bin == 1).sum()

xgb_final = XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.5,
    scale_pos_weight=sw_ratio,
    objective="binary:logistic", eval_metric="aucpr",
    random_state=42, n_jobs=-1, tree_method="hist",
)
xgb_final.fit(X_all, y_bin, verbose=False)

print("=== Top-30 特征重要性 ===")
idx = np.argsort(xgb_final.feature_importances_)[::-1][:30]
for rank, i in enumerate(idx):
    val = xgb_final.feature_importances_[i]
    bar = "█" * int(val * 2000)
    print(f"  {rank+1:2d}. {feat_cols[i]:25s}  {val:.5f}  {bar}")


=== Top-30 特征重要性 ===
   1. esm_73                     0.00492  █████████
   2. esm_998                    0.00426  ████████
   3. esm_219                    0.00392  ███████
   4. esm_987                    0.00384  ███████
   5. esm_838                    0.00382  ███████
   6. esm_661                    0.00370  ███████
   7. esm_1191                   0.00356  ███████
   8. esm_1126                   0.00351  ███████
   9. esm_495                    0.00344  ██████
  10. esm_986                    0.00336  ██████
  11. esm_69                     0.00333  ██████
  12. esm_567                    0.00326  ██████
  13. esm_384                    0.00326  ██████
  14. esm_440                    0.00322  ██████
  15. esm_64                     0.00318  ██████
  16. esm_1188                   0.00304  ██████
  17. esm_60                     0.00303  ██████
  18. esm_377                    0.00302  ██████
  19. esm_1270                   0.00301  ██████
  20. esm_868                    0.00